# **CardiAg: Cardio-Respiratory Phase Analysis**

Pipeline for the analysis of behavioral events from the CardiAg Intentional Binding task with respect to bimodal cardio-respiratory phase derived from the electrocardiography (ECG) and respiration belt (RESP) signals. 

Authors: Marta Gerosa

Created on: 22 July 2025

Last update (by M. Gerosa): 21 April 2026

Before using the pipeline, we recommend creating the dedicated `bbsig_pipeline` environment and conducting the unimodal cardiac phase & respiratory phase analyses using the corresponding `.ipynb` notebooks. This will generate the required files in the `/derivatives/cardiac-phase-analysis/` and `/derivatives/resp-phase-analysis/` folders. 

## **Pipeline structure**

The following steps are included:

1. **Cardiac & respiratory phase data import & merge**: import the dataframes created during the cardiac phase analysis (`task-CardiAgIBTask_beh_ecg_{action/tone}.tsv`) and respiratory phase analysis (`task-CardiAgIBTask_beh_resp_{action/tone}.tsv`) steps, and merge them together according to the respective task event, i.e. action or tone. The merged files `task-CardiAgIBTask_beh_ecg_resp_action.tsv` and `task-CardiAgIBTask_beh_ecg_resp_tone.tsv` are saved in the `derivatives/cardioresp-phase-analysis/` directory. 
2. **Cardiac & respiratory phase effects on intentional binding [Results 2.4]**: analyze differences in a dependent variable (e.g., action/tone binding) according to whether a task event onset was binned in systole vs. diastole, & expiration vs. inspiration. Then, a 2x2 repeated-measure ANOVA is used to compare differences in the intentional binding measures (i.e., action or tone binding) between the different cardio-respiratory phases. This corresponds to **Results section 2.4,** with associated **Figure 5c**. 
3. **Phase Locking Value (PLV) analysis [Results 2.4]**: calculate the phase-locking value (PLV) between cardiac and respiratory phase at action/tone onset, using n:m synchronization (Lachaux et al., 1999; Tass et al., 1998), following the method implemented in Grund et al. (2022). PLVs provide information about the extent to which the weighted phase-difference of the two signals at the time of the task event (i.e., action onset or tone delivery) stays constant over trials, ranging from 0 (no intertrial coupling between the cardiac and respiratory signals) to 1 (constant intertrial coupling). Then, separately for the time of action and tone onset, PLVs are compared between Baseline and Operant conditions using paired-sample t-tests. This corresponds to **Results section 2.4**. 
4. **R-R interval changes across respiratory cycle [Suppl. Results 1.2.4]**: provide descriptives for the cardio-respiratory interaction, namely changes in R-R interval duration across the respiratory cycle, which are analysed using a rmANOVA with 8 bins (45° each) relative to the first 0-45° bin. This is reported in **Supplementary Results 1.2.4**, with corresponding **Figure S3c**. 

In [1]:
############## Import modules ##############

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from IPython.display import Markdown as md
import pingouin as pg

In [2]:
############## Settings: optional pipeline steps ##############

# For all sections: choose the task event type that will be analysed with respect to the respiratory cycle,
# by specifying the column name with timestamps (in sec) and the abbreviation to be appended in the variable names

# Action onset analysis
event_col = 'keypress_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev = 'act'                  # define abbreviation of task event for further column names in respiratory phase analysis

# Tone onset analysis
event_col_tone = 'tone_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev_tone = 'tone'             # define abbreviation of task event for further column names in respiratory phase analysis

# For all sections: specify your column names for participant ID, condition, trial and block (the latter is optional)
column_map = {
    'participant': 'subjID',
    'condition': 'condition',
    'trial': 'n_trial',
    'block': 'n_block' # remove this key-value pair if block is not available
}

## **1. Cardiac & respiratory phase data import & merge**

This section imports the dataframes created during the cardiac phase analysis (`task-CardiAgIBTask_beh_ecg_{action/tone}.tsv`) and respiratory phase analysis (`task-CardiAgIBTask_beh_resp_{action/tone}.tsv`) steps, and merges them together according to the respective task event, i.e. action or tone. Furthermore, the directory for storing the results of the bimodal cardio-respiratory phase analysis (`derivatives/cardioresp-phase-analysis/`) is created. 

In [ ]:
################## 1a. Cardiac & respiratory phase data import ##################

# Sub-16 excluded due to extreme JE distribution, sub-14 excluded due to missing HW triggers
# Sub-33 excluded due to outlier IB measures (similar to ECG analysis)
participant_ids = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19, 
                   20, 21, 22, 23, 24, 25, 26, 27, 28, 30,          # removed: sub-29 (bad RESP signal quality)
                   31, 32, 34, 35, 36, 37, 38, 39, 41,              # removed: sub-42 (bad RESP signal quality)
                   43, 44, 45, 47, 48, 49, 51, 53, 54, 55, 57]      # removed: sub-46 (bad RESP signal quality)

# Specify the data path info
wd = r'.\data'              # directory of data storage
exp_name = 'CardiAgIBTask' 
datatype_name = 'beh'       # current datatype folder according to BIDS

for event_name in ['action', 'tone']: 

    ####### Load dataframe from cardiac phase analysis - action & tone onset #######

    # Check if the BIDS directory 'derivatives/cardiac-phase-analysis/' exists
    cardphase_folder = 'cardiac-phase-analysis'
    cardphase_dir = os.path.join(wd, 'derivatives', cardphase_folder)
    if not os.path.isdir(cardphase_dir):
        print(f"Directory does not exist: {cardphase_dir}")

    # Check if file 'task-CardiAgIBTask_beh_ecg_{event_name}.tsv' exists, and if so save corrisponding df
    beh_ecg_event_dir = os.path.join(cardphase_dir, f'task-CardiAgIBTask_beh_ecg_{event_name}.tsv')

    if not os.path.isfile(beh_ecg_event_dir):
        print(f"File does not exist: {beh_ecg_event_dir}")
    else:
        df = pd.read_csv(beh_ecg_event_dir, sep='\t', header=0)

        # Save cardiac phase analysis df for action/tone trials and filter for excluded participants
        if event_name == 'action':
            beh_ecg_action = df
            beh_ecg_action = beh_ecg_action[beh_ecg_action['subjID'].isin(participant_ids)]
        elif event_name == 'tone':
            beh_ecg_tone = df
            beh_ecg_tone = beh_ecg_tone[beh_ecg_tone['subjID'].isin(participant_ids)]
    

    ####### Load dataframe from respiratory phase analysis - action & tone onset #######

    # Check if the BIDS directory 'derivatives/resp-phase-analysis/' exists
    respphase_folder = 'resp-phase-analysis'
    respphase_dir = os.path.join(wd, 'derivatives', respphase_folder)
    if not os.path.isdir(respphase_dir):
        print(f"Directory does not exist: {respphase_dir}")

    # Check if file 'task-CardiAgIBTask_beh_resp_{event_name}.tsv' exists, and if so save corrisponding df
    beh_resp_event_dir = os.path.join(respphase_dir, f'task-CardiAgIBTask_beh_resp_{event_name}.tsv')

    if not os.path.isfile(beh_resp_event_dir):
        print(f"File does not exist: {beh_resp_event_dir}")
    else:
        df = pd.read_csv(beh_resp_event_dir, sep='\t', header=0)

        # Save respiratory phase analysis df for action/tone trials and filter for excluded participants
        if event_name == 'action':
            beh_resp_action = df
            beh_resp_action = beh_resp_action[beh_resp_action['subjID'].isin(participant_ids)]
        elif event_name == 'tone':
            beh_resp_tone = df
            beh_resp_tone = beh_resp_tone[beh_resp_tone['subjID'].isin(participant_ids)]
    

####### Create general directory for saving output from cardiac-respiratory phase bimodal analysis #######

# Check if the BIDS directory 'derivatives/cardioresp-phase-analysis/' exists, if not create it
ecg_resp_folder = 'cardioresp-phase-analysis'
ecg_resp_dir = os.path.join(wd, 'derivatives', ecg_resp_folder)
if not os.path.exists(ecg_resp_dir):
    os.makedirs(ecg_resp_dir)

In [4]:
################## 1b. Cardiac & respiratory phase data merge ##################

# Concatenate dfs for cardiac & respiratory phase analysis - action onset
common_cols_act = beh_ecg_action.columns.intersection(beh_resp_action.columns)
beh_resp_action_unique = beh_resp_action.drop(columns=common_cols_act) # drop overlapping columns

# Concatenate along columns (axis=1)
beh_ecg_resp_action = pd.concat([beh_ecg_action.reset_index(drop=True), 
                                 beh_resp_action_unique.reset_index(drop=True)], axis=1)

# Save output file as TSV in 'derivatives/cardioresp-phase-analysis' directory
beh_ecg_resp_action_dir = os.path.join(ecg_resp_dir, 'task-CardiAgIBTask_beh_ecg_resp_action.tsv')
beh_ecg_resp_action.to_csv(beh_ecg_resp_action_dir, index=False, sep='\t', na_rep="n/a")

# Concatenate dfs for cardiac & respiratory phase analysis - tone onset
common_cols_tone = beh_ecg_tone.columns.intersection(beh_resp_tone.columns)
beh_resp_tone_unique = beh_resp_tone.drop(columns=common_cols_tone) # drop overlapping columns

# Concatenate along columns (axis=1)
beh_ecg_resp_tone = pd.concat([beh_ecg_tone.reset_index(drop=True), 
                               beh_resp_tone_unique.reset_index(drop=True)], axis=1)

# Save output file as TSV in 'derivatives/cardioresp-phase-analysis' directory
beh_ecg_resp_tone_dir = os.path.join(ecg_resp_dir, 'task-CardiAgIBTask_beh_ecg_resp_tone.tsv')
beh_ecg_resp_tone.to_csv(beh_ecg_resp_tone_dir, index=False, sep='\t', na_rep="n/a")

In [5]:
# Create directory for saving plots
project_root = os.path.dirname(wd)
plotting_dir = os.path.join(project_root, 'results', 'datavisualization')
if not os.path.exists(plotting_dir):
    os.makedirs(plotting_dir)

## **2. Cardiac & respiratory phase effects on intentional binding [Results 2.4]**

This section defines a function to analyze differences in a dependent variable (e.g., action binding) according to whether a task event onset was binned in systole vs. diastole, & expiration vs. inspiration. Then, a 2x2 repeated-measure ANOVA is used to compare differences in the intentional binding measures (i.e., action or tone binding) between the different cardio-respiratory phases. In detail:
 
- Create subsets at the combination of cardiac and respiratory phase
- Perform the 2x2 rmANOVA of action/tone binding between cardiac (systole/diastole) and respiratory (expiration/inspiration) phases
- Summary arrow plot of action & tone binding across cardio-respiratory phases (Figure 5c)
- Save the averages for action and tone binding as `task-CardiAgIBTask_ecg_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_resp_tonebinding_avg.tsv`, respectively, in the `derivatives/cardioresp-phase-analysis` directory

In [6]:
################## 2. Cardiac & respiratory phase effects on dependent variable (e.g., action binding) ##################

# Define function to analyze differences in a dependent variable (e.g., action binding) according to 
# whether a task event onset was binned in systole vs. diastole, and expiration vs. inspiration. 

def analyze_cardresp_binding(beh_ecg_resp_df, conditions, abbrev, depend_var,
                             participant_col=column_map['participant'], condition_col=column_map['condition']):

    # Compute means per cell
    def compute_cell(df, cardiac_label, resp_label):

        df = df[df[condition_col].isin(conditions)] # Filter only selected conditions

        means = (df.groupby([participant_col, condition_col])[depend_var].mean().unstack())
        means[f'{abbrev}_binding'] = means[conditions[1]] - means[conditions[0]] # Compute binding 
        means = means[[f'{abbrev}_binding']].reset_index()
        
        means['cardiac'] = cardiac_label
        means['resp'] = resp_label

        return means

    # Create four cardiac x respiratory phase subsets
    sys_exp = beh_ecg_resp_df[(beh_ecg_resp_df[f'{abbrev}_sys'] == 1) & (beh_ecg_resp_df[f'{abbrev}_exp'] == 1)]
    sys_insp = beh_ecg_resp_df[(beh_ecg_resp_df[f'{abbrev}_sys'] == 1) & (beh_ecg_resp_df[f'{abbrev}_insp'] == 1)]
    dia_exp = beh_ecg_resp_df[(beh_ecg_resp_df[f'{abbrev}_dia'] == 1) & (beh_ecg_resp_df[f'{abbrev}_exp'] == 1)]
    dia_insp = beh_ecg_resp_df[(beh_ecg_resp_df[f'{abbrev}_dia'] == 1) & (beh_ecg_resp_df[f'{abbrev}_insp'] == 1)]

    # Compute binding for each subset
    df_sys_exp  = compute_cell(sys_exp,  'systole',  'expiration')
    df_sys_insp = compute_cell(sys_insp, 'systole',  'inspiration')
    df_dia_exp  = compute_cell(dia_exp,  'diastole', 'expiration')
    df_dia_insp = compute_cell(dia_insp, 'diastole', 'inspiration')

    # Combine into long format for factorial ANOVA
    df_long = pd.concat([df_sys_exp, df_sys_insp, df_dia_exp, df_dia_insp], ignore_index=True)

    # 2×2 repeated-measures ANOVA
    rmANOVA_res = pg.rm_anova(dv=f'{abbrev}_binding',
                              within=['cardiac', 'resp'], # cardiac & respiratory phase as within factors
                              subject=participant_col,
                              data=df_long, detailed=True)

    return df_long, rmANOVA_res

In [7]:
# Analyze action binding according to action onset in cardiac/respiratory phase (2x2 rmANOVA)
df_long_act, rmANOVA_actbind = analyze_cardresp_binding(beh_ecg_resp_df=beh_ecg_resp_action, 
                                                        conditions=('BasA', 'OpA'), 
                                                        abbrev=abbrev, depend_var='JE_act')

print(rmANOVA_actbind)

           Source           SS  ddof1  ddof2           MS         F     p_unc  \
0         cardiac  9232.385989      1     40  9232.385989  3.271596  0.078010   
1            resp    81.612889      1     40    81.612889  0.046229  0.830852   
2  cardiac * resp  4522.769232      1     40  4522.769232  2.276354  0.139219   

   p_GG_corr       ng2  eps  
0   0.078010  0.018640  1.0  
1   0.830852  0.000168  1.0  
2   0.139219  0.009219  1.0  


In [10]:
# Function to convert output from rmANOVA into APA-like text
def apa_rm_anova(anova_res, effect):
    
    row = anova_res.query("Source == @effect").iloc[0]
    
    df1 = int(row['ddof1'])
    df2 = int(row['ddof2'])
    F   = round(row['F'], 3)
    
    # Use GG correction if available
    if 'p-GG-corr' in row:
        p = row['p-GG-corr']
    else:
        p = row['p_unc']
        
    p = round(p, 3)
    eta = round(row['ng2'], 3)

    return f"F({df1}, {df2}) = {F}, p = {p}, η² = {eta}"

In [11]:
# Save APA-like text from 2x2 rmANOVA of action binding
actbind_card = apa_rm_anova(rmANOVA_actbind, "cardiac")
actbind_resp = apa_rm_anova(rmANOVA_actbind, "resp")
actbind_inter = apa_rm_anova(rmANOVA_actbind, "cardiac * resp")

# Calculate summary descriptives of action binding according to cardiac & respiratory phase
actbind_sum = (df_long_act.groupby(['cardiac','resp'])[f'{abbrev}_binding']
                 .agg(['mean','std']).round(2))

md(f"""A 2×2 repeated-measures ANOVA tested the effect of cardiac phase (systole vs. diastole) 
   and respiratory phase (expiration vs. inspiration) at action onset on action binding. 
   There was no significant main effect of cardiac phase ({actbind_card}) nor of respiratory phase ({actbind_resp}).
   The cardiac × respiratory interaction was not significant ({actbind_inter}).

   Action binding was not different depending on whether the action occurred at 
   systole–expiration (M = {actbind_sum.loc[('systole','expiration'),'mean']}, SD = {actbind_sum.loc[('systole','expiration'),'std']}),
   systole–inspiration (M = {actbind_sum.loc[('systole','inspiration'),'mean']}, SD = {actbind_sum.loc[('systole','inspiration'),'std']}),
   diastole–expiration (M = {actbind_sum.loc[('diastole','expiration'),'mean']}, SD = {actbind_sum.loc[('diastole','expiration'),'std']}),
   or diastole–inspiration (M = {actbind_sum.loc[('diastole','inspiration'),'mean']}, SD = {actbind_sum.loc[('diastole','inspiration'),'std']}).""")

A 2×2 repeated-measures ANOVA tested the effect of cardiac phase (systole vs. diastole) 
   and respiratory phase (expiration vs. inspiration) at action onset on action binding. 
   There was no significant main effect of cardiac phase (F(1, 40) = 3.272, p = 0.078, η² = 0.019) nor of respiratory phase (F(1, 40) = 0.046, p = 0.831, η² = 0.0).
   The cardiac × respiratory interaction was not significant (F(1, 40) = 2.276, p = 0.139, η² = 0.009).

   Action binding was not different depending on whether the action occurred at 
   systole–expiration (M = 38.99, SD = 46.16),
   systole–inspiration (M = 50.9, SD = 81.86),
   diastole–expiration (M = 34.49, SD = 39.14),
   or diastole–inspiration (M = 25.39, SD = 42.28).

In [12]:
# Calculate post-hoc comparisons for 2x2 rmANOVA of action binding
actbind_posthoc = pg.pairwise_tests(dv=f'{abbrev}_binding', within=['cardiac','resp'],
                            subject=column_map['participant'],
                            data=df_long_act, padjust='bonf')

print(actbind_posthoc)

         Contrast   cardiac           A            B Paired Parametric  \
0         cardiac         -    diastole      systole   True       True   
1            resp         -  expiration  inspiration   True       True   
2  cardiac * resp  diastole  expiration  inspiration   True       True   
3  cardiac * resp   systole  expiration  inspiration   True       True   

          T   dof alternative     p_unc    p_corr p_adjust   BF10    hedges  
0 -1.808755  40.0   two-sided  0.078010       NaN      NaN  0.745 -0.324395  
1 -0.215010  40.0   two-sided  0.830852       NaN      NaN  0.172 -0.032620  
2  1.198022  40.0   two-sided  0.237959  0.475918     bonf  0.328  0.221083  
3 -1.063731  40.0   two-sided  0.293830  0.587660     bonf  0.286 -0.177594  


In [13]:
# Analyze tone binding according to tone onset in cardiac/respiratory phase (2x2 rmANOVA)
df_long_tone, rmANOVA_tonebind = analyze_cardresp_binding(beh_ecg_resp_df=beh_ecg_resp_tone, 
                                                          conditions=('BasT', 'OpT'), 
                                                          abbrev=abbrev_tone, depend_var='JE_tone')

print(rmANOVA_tonebind)

           Source            SS  ddof1  ddof2            MS         F  \
0         cardiac    699.910970      1     40    699.910970  0.304626   
1            resp    458.096502      1     40    458.096502  0.259137   
2  cardiac * resp  20592.082018      1     40  20592.082018  6.886100   

      p_unc  p_GG_corr       ng2  eps  
0  0.584068   0.584068  0.000527  1.0  
1  0.613511   0.613511  0.000345  1.0  
2  0.012240   0.012240  0.015285  1.0  


In [14]:
# Save APA-like text from 2x2 rmANOVA of tone binding
tonebind_card = apa_rm_anova(rmANOVA_tonebind, "cardiac")
tonebind_resp = apa_rm_anova(rmANOVA_tonebind, "resp")
tonebind_inter = apa_rm_anova(rmANOVA_tonebind, "cardiac * resp")

# Calculate summary descriptives of tone binding according to cardiac & respiratory phase
tonebind_sum = (df_long_tone.groupby(['cardiac','resp'])[f'{abbrev_tone}_binding']
                 .agg(['mean','std']).round(2))

md(f"""A 2×2 repeated-measures ANOVA tested the effect of cardiac phase (systole vs. diastole) 
   and respiratory phase (expiration vs. inspiration) at tone onset on tone binding. 
   There was no significant main effect of cardiac phase ({tonebind_card}) nor of respiratory phase ({tonebind_resp}).
   However, the cardiac × respiratory interaction was significant ({tonebind_inter}).

   Tone binding differed as a function of the combination of cardiac and respiratory phase. 
   During diastole, tone binding was more negative during inspiration (M = {tonebind_sum.loc[('diastole','inspiration'),'mean']}, 
   SD = {tonebind_sum.loc[('diastole','inspiration'),'std']}) than expiration (M = {tonebind_sum.loc[('diastole','expiration'),'mean']}, 
   SD = {tonebind_sum.loc[('diastole','expiration'),'std']}), whereas during systole the opposite pattern was observed, with more negative
   tone binding during expiration (M = {tonebind_sum.loc[('systole','expiration'),'mean']}, SD = {tonebind_sum.loc[('systole','expiration'),'std']})
   than inspiration (M = {tonebind_sum.loc[('systole','inspiration'),'mean']}, SD = {tonebind_sum.loc[('systole','inspiration'),'std']}).""")

A 2×2 repeated-measures ANOVA tested the effect of cardiac phase (systole vs. diastole) 
   and respiratory phase (expiration vs. inspiration) at tone onset on tone binding. 
   There was no significant main effect of cardiac phase (F(1, 40) = 0.305, p = 0.584, η² = 0.001) nor of respiratory phase (F(1, 40) = 0.259, p = 0.614, η² = 0.0).
   However, the cardiac × respiratory interaction was significant (F(1, 40) = 6.886, p = 0.012, η² = 0.015).

   Tone binding differed as a function of the combination of cardiac and respiratory phase. 
   During diastole, tone binding was more negative during inspiration (M = -98.64, 
   SD = 88.28) than expiration (M = -72.89, 
   SD = 80.98), whereas during systole the opposite pattern was observed, with more negative
   tone binding during expiration (M = -91.17, SD = 87.83)
   than inspiration (M = -72.1, SD = 105.36).

In [15]:
# Calculate post-hoc comparisons for 2x2 rmANOVA of tone binding
tonebind_posthoc = pg.pairwise_tests(dv=f'{abbrev_tone}_binding', within=['cardiac','resp'],
                            subject=column_map['participant'],
                            data=df_long_tone, padjust='bonf')

print(tonebind_posthoc)

         Contrast   cardiac           A            B Paired Parametric  \
0         cardiac         -    diastole      systole   True       True   
1            resp         -  expiration  inspiration   True       True   
2  cardiac * resp  diastole  expiration  inspiration   True       True   
3  cardiac * resp   systole  expiration  inspiration   True       True   

          T   dof alternative     p_unc    p_corr p_adjust   BF10    hedges  
0 -0.551929  40.0   two-sided  0.584068       NaN      NaN  0.195 -0.048566  
1  0.509055  40.0   two-sided  0.613511       NaN      NaN  0.191  0.039662  
2  2.649516  40.0   two-sided  0.011487  0.022973     bonf  3.591  0.301170  
3 -1.625400  40.0   two-sided  0.111932  0.223864     bonf  0.564 -0.194743  


In [16]:
# Save average files for action/tone binding as TSV in 'derivatives/cardioresp-phase-analysis' directory
actbind_dir = os.path.join(ecg_resp_dir, 'task-CardiAgIBTask_ecg_resp_actbinding_avg.tsv')
df_long_act.to_csv(actbind_dir, index=False, sep='\t', na_rep="n/a")

tonebind_dir = os.path.join(ecg_resp_dir, 'task-CardiAgIBTask_ecg_resp_tonebinding_avg.tsv')
df_long_tone.to_csv(tonebind_dir, index=False, sep='\t', na_rep="n/a")

In [17]:
# Define function to compute cardio-respiratory phase averages
def compute_phase_means(beh_ecg_resp_df, abbrev):

    phases = {"sys_exp": (beh_ecg_resp_df[f"{abbrev}_sys"]==1) & (beh_ecg_resp_df[f"{abbrev}_exp"]==1),
              "sys_insp": (beh_ecg_resp_df[f"{abbrev}_sys"]==1) & (beh_ecg_resp_df[f"{abbrev}_insp"]==1),
              "dia_exp": (beh_ecg_resp_df[f"{abbrev}_dia"]==1) & (beh_ecg_resp_df[f"{abbrev}_exp"]==1),
              "dia_insp": (beh_ecg_resp_df[f"{abbrev}_dia"]==1) & (beh_ecg_resp_df[f"{abbrev}_insp"]==1)}

    output = {}

    for p, mask in phases.items():

        tmp = beh_ecg_resp_df[mask]
        means = (tmp.groupby(["subjID","condition"]).mean(numeric_only=True).reset_index())

        if abbrev == "act":
            output[p] = {"BasA": means.loc[means["condition"]=="BasA","JE_act"].mean(),
                         "OpA":  means.loc[means["condition"]=="OpA","JE_act"].mean()}

        elif abbrev == "tone":
            output[p] = {"BasT": means.loc[means["condition"]=="BasT","JE_tone"].mean(),
                         "OpT":  means.loc[means["condition"]=="OpT","JE_tone"].mean()}

    return output

# Calculate means for action/tone bindinding according to cardio-respiratory phase
actbind_means = compute_phase_means(beh_ecg_resp_df=beh_ecg_resp_action, abbrev=abbrev)
tonebind_means = compute_phase_means(beh_ecg_resp_df=beh_ecg_resp_tone, abbrev=abbrev_tone)

print(f"----- Action binding means -----\n{actbind_means}")
print(f"\n----- Tone binding means -----\n{tonebind_means}")

----- Action binding means -----
{'sys_exp': {'BasA': 25.994632509289413, 'OpA': 64.98293409833543}, 'sys_insp': {'BasA': 12.788151614369038, 'OpA': 63.69025019124487}, 'dia_exp': {'BasA': 25.23491181589103, 'OpA': 59.72013526687992}, 'dia_insp': {'BasA': 28.839195837695858, 'OpA': 54.2323652166647}}

----- Tone binding means -----
{'sys_exp': {'BasT': 48.42155104330113, 'OpT': -42.74850741959953}, 'sys_insp': {'BasT': 42.76011213833779, 'OpT': -29.341719456151278}, 'dia_exp': {'BasT': 37.46617129725945, 'OpT': -35.424750324150196}, 'dia_insp': {'BasT': 47.919925350698655, 'OpT': -50.72445624581512}}


In [ ]:
############## Summary arrow plot of action & tone binding across cardio-respiratory phases ##############

### FIGURE 5c ###

colpalette = ['#007a4e','#a4cbb6','#0b3142','#7796cb']

fig, ax = plt.subplots(figsize=(7.5, 3), dpi=300)

# Draw a horizontal line and labels for time
time_axis = patches.FancyArrowPatch((-50, 0.8), (320, 0.8), color='k', arrowstyle='-|>', mutation_scale=8)
ax.add_patch(time_axis)
ax.text(x=125, y=0.83, s='250 ms', color='k', ha='center', va='center', fontsize=9) # delay label
ax.text(x=310, y=0.76, s='Time', color='k', ha='right', va='center', fontsize=9) # time axis label
int_arrow = patches.FancyArrowPatch((0, 0.77), (250, 0.77), color='grey', arrowstyle='<|-|>', 
                                    mutation_scale=8, linewidth=0.8)
ax.add_patch(int_arrow)
ax.text(x=125, y=0.74, s='Actual interval', color='grey', ha='center', va='center', fontsize=7) # actual interval label

# Draw line for actual Action time (0 ms)
ax.axvline(x=0, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=0, y=0.92, s='Action', color='k', ha='center', va='center', fontsize=13)

# Draw line for actual Tone time (250ms)
ax.axvline(x=250, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=250, y=0.92, s='Tone', color='k', ha='center', va='center', fontsize=13)

# Draw perceived Baseline/Operant times for SYS-EXP 
ax.axvline(x=actbind_means["sys_exp"]["BasA"], color=colpalette[1], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.axvline(x=(250 + tonebind_means["sys_exp"]["BasT"]), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.axvline(x=actbind_means["sys_exp"]["OpA"], color=colpalette[0], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.axvline(x=(250 + tonebind_means["sys_exp"]["OpT"]), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.text(x=-70, y=0.65, s='SYS-EXP', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Baseline/Operant times for SYS-INS
ax.axvline(x=actbind_means["sys_insp"]["BasA"], color=colpalette[1], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.axvline(x=(250 + tonebind_means["sys_insp"]["BasT"]), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.axvline(x=actbind_means["sys_insp"]["OpA"], color=colpalette[0], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.axvline(x=(250 + tonebind_means["sys_insp"]["OpT"]), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.text(x=-70, y=0.5, s='SYS-INS', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Baseline/Operant times for DIA-EXP
ax.axvline(x=actbind_means["dia_exp"]["BasA"], color=colpalette[1], linestyle='-', linewidth=4, ymin=0.3, ymax=0.4)
ax.axvline(x=(250 + tonebind_means["dia_exp"]["BasT"]), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.3, ymax=0.4)
ax.axvline(x=actbind_means["dia_exp"]["OpA"], color=colpalette[0], linestyle='-', linewidth=4, ymin=0.3, ymax=0.4)
ax.axvline(x=(250 + tonebind_means["dia_exp"]["OpT"]), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.3, ymax=0.4)
ax.text(x=-70, y=0.35, s='DIA-EXP', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Baseline/Operant times for DIA-EXP
ax.axvline(x=actbind_means["dia_insp"]["BasA"], color=colpalette[1], linestyle='-', linewidth=4, ymin=0.15, ymax=0.25)
ax.axvline(x=(250 + tonebind_means["dia_insp"]["BasT"]), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.15, ymax=0.25)
ax.axvline(x=actbind_means["dia_insp"]["OpA"], color=colpalette[0], linestyle='-', linewidth=4, ymin=0.15, ymax=0.25)
ax.axvline(x=(250 + tonebind_means["dia_insp"]["OpT"]), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.15, ymax=0.25)
ax.text(x=-70, y=0.2, s='DIA-INS', color='k', ha='left', va='center', fontsize=11)

# Draw arrow and add text for Action Binding across all cardio-respiratory groups
ax.arrow(x=(actbind_means["sys_exp"]["BasA"]+1), y=0.65, dx=(actbind_means["sys_exp"]["OpA"] - actbind_means["sys_exp"]["BasA"])*0.88, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means["sys_exp"]["BasA"] + (actbind_means["sys_exp"]["OpA"] - actbind_means["sys_exp"]["BasA"])/2)*0.95, y=0.62, 
        s=f"{actbind_sum.loc[('systole','expiration'),'mean']:.2f} ({actbind_sum.loc[('systole','expiration'),'std']:.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(actbind_means["sys_insp"]["BasA"]+1), y=0.5, dx=(actbind_means["sys_insp"]["OpA"] - actbind_means["sys_insp"]["BasA"])*0.91, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means["sys_insp"]["BasA"] + (actbind_means["sys_insp"]["OpA"] - actbind_means["sys_insp"]["BasA"])/2)*0.95, y=0.47, 
        s=f"{actbind_sum.loc[('systole','inspiration'),'mean']:.2f} ({actbind_sum.loc[('systole','inspiration'),'std']:.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(actbind_means["dia_exp"]["BasA"]+1), y=0.35, dx=(actbind_means["dia_exp"]["OpA"] - actbind_means["dia_exp"]["BasA"])*0.88, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means["dia_exp"]["BasA"] + (actbind_means["dia_exp"]["OpA"].mean() - actbind_means["dia_exp"]["BasA"])/2)*0.9, y=0.32, 
        s=f"{actbind_sum.loc[('diastole','expiration'),'mean']:.2f} ({actbind_sum.loc[('diastole','expiration'),'std']:.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(actbind_means["dia_insp"]["BasA"]+1), y=0.2, dx=(actbind_means["dia_insp"]["OpA"] - actbind_means["dia_insp"]["BasA"])*0.82, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means["dia_insp"]["BasA"] + (actbind_means["dia_insp"]["OpA"] - actbind_means["dia_insp"]["BasA"])/2)*0.8, y=0.17, 
        s=f"{actbind_sum.loc[('diastole','inspiration'),'mean']:.2f} ({actbind_sum.loc[('diastole','inspiration'),'std']:.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)

# Draw arrow and add text for Tone Binding across all cardio-respiratory groups
ax.arrow(x=(250 + tonebind_means["sys_exp"]["BasT"]), y=0.65, dx=(tonebind_means["sys_exp"]["OpT"] - tonebind_means["sys_exp"]["BasT"])*0.95, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means["sys_exp"]["BasT"])+((tonebind_means["sys_exp"]["OpT"] - tonebind_means["sys_exp"]["BasT"])/2)*0.95), y=0.62, 
         s=f"{tonebind_sum.loc[('systole','expiration'),'mean']:.2f} ({tonebind_sum.loc[('systole','expiration'),'std']:.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)
ax.arrow(x=(250 + tonebind_means["sys_insp"]["BasT"]), y=0.5, dx=(tonebind_means["sys_insp"]["OpT"] - tonebind_means["sys_insp"]["BasT"])*0.935, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means["sys_insp"]["BasT"])+((tonebind_means["sys_insp"]["OpT"] - tonebind_means["sys_insp"]["BasT"])/2)*0.95), y=0.47,  
         s=f"{tonebind_sum.loc[('systole','inspiration'),'mean']:.2f} ({tonebind_sum.loc[('systole','inspiration'),'std']:.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)
ax.arrow(x=(250 + tonebind_means["dia_exp"]["BasT"]), y=0.35, dx=(tonebind_means["dia_exp"]["OpT"] - tonebind_means["dia_exp"]["BasT"])*0.935, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means["dia_exp"]["BasT"])+((tonebind_means["dia_exp"]["OpT"] - tonebind_means["dia_exp"]["BasT"])/2)*0.95), y=0.32,  
         s=f"{tonebind_sum.loc[('diastole','expiration'),'mean']:.2f} ({tonebind_sum.loc[('diastole','expiration'),'std']:.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)
ax.arrow(x=(250 + tonebind_means["dia_insp"]["BasT"]), y=0.2, dx=(tonebind_means["dia_insp"]["OpT"] - tonebind_means["dia_insp"]["BasT"])*0.95, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means["dia_insp"]["BasT"])+((tonebind_means["dia_insp"]["OpT"] - tonebind_means["dia_insp"]["BasT"])/2)*0.95), y=0.17,  
         s=f"{tonebind_sum.loc[('diastole','inspiration'),'mean']:.2f} ({tonebind_sum.loc[('diastole','inspiration'),'std']:.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)

# Set limits and remove axes
ax.set_xlim(-100, 350)
ax.set_ylim(0, 1)
ax.axis('off')
plt.tight_layout()

# Save plot 
fig9 = os.path.join(plotting_dir, "Figure5c_cardiorespiratory_arrow_summary.svg")
plt.savefig(fig9, format="svg")
plt.show()

## **3. Phase Locking Value (PLV) analysis [Results 2.4]**

This section calculates the phase-locking value (PLV) between cardiac and respiratory phase at action/tone onset, using n:m synchronization (Lachaux et al., 1999; Tass et al., 1998), following the method implemented in Grund et al. (2022). PLVs provide information about the extent to which the weighted phase-difference of the two signals at the time of the task event (i.e., action onset or tone delivery) stays constant over trials, ranging from 0 (no intertrial coupling between the cardiac and respiratory signals) to 1 (constant intertrial coupling). Then, separately for the time of action and tone onset, PLVs are compared between Baseline and Operant conditions using paired-sample t-tests. This is reported in Results section 2.4. 

In [19]:
##### 4. Phase-locking between cardiac and respiratory activity - n:m synchronization #####

# Define function to calculate phase-locking value (PLV) of cardiac and respiratory phase at time of action/tone onset, 
# using n:m synchronization
def compute_ecg_resp_plv(beh_ecg_resp_df, abbrev, 
                         participant_col=column_map['participant'], condition_col=column_map['condition']): 

    # Compute mean cardiac and respiratory frequency per participant
    mean_cardioresp_freq = beh_ecg_resp_df.groupby(participant_col).agg(
        mean_cardiac_rate = (f'HR_1perSec_{abbrev}', 'mean'),
        mean_resp_rate = (f'RR_resp_1perSec_{abbrev}', 'mean')).reset_index()

    # Compute ratio between mean respiratory and cardiac frequency (rounded) -> m value
    mean_cardioresp_freq['cardioresp_ratio_m'] = np.round(mean_cardioresp_freq['mean_cardiac_rate'] / mean_cardioresp_freq['mean_resp_rate'])
    beh_ecg_resp_df = beh_ecg_resp_df.merge(mean_cardioresp_freq[[participant_col, 'cardioresp_ratio_m']], on=participant_col)

    # Compute weighted phase difference Δφ = (m * event_resp_rad_{abbrev} - event_ecg_radian_{abbrev})
    beh_ecg_resp_df['cardioresp_delta_phase'] = (
        beh_ecg_resp_df['cardioresp_ratio_m'] * beh_ecg_resp_df[f'event_resp_rad_{abbrev}'] - beh_ecg_resp_df[f'event_ecg_radian_{abbrev}'])

    # Create exponential (convert cardioresp_delta_phase to unit complex numbers)
    beh_ecg_resp_df['cardioresp_delta_phase_cpl'] = np.exp(1j * beh_ecg_resp_df['cardioresp_delta_phase'])

    # Compute PLV per subject and condition
    plv_df = (
        beh_ecg_resp_df
        .groupby([participant_col, condition_col])['cardioresp_delta_phase_cpl']
        .apply(lambda x: np.abs(np.mean(x)))
        .reset_index(name='PLV'))
    
    # Average PLV across participants
    mean_plv = plv_df['PLV'].mean()
    print(f"Mean PLV across participants and conditions for {abbrev} trials: {mean_plv:.4f}")

    return beh_ecg_resp_df, plv_df 

In [20]:
# Compute cardio-respiratory PLV at action onset (BasA, OpA)
beh_ecg_resp_action_filter = beh_ecg_resp_action[beh_ecg_resp_action[column_map['condition']].isin(['BasA', 'OpA'])]
beh_ecg_resp_tone_filter = beh_ecg_resp_tone[beh_ecg_resp_tone[column_map['condition']].isin(['BasT', 'OpT'])]

# Compute cardio-respiratory PLV at tone onset (BasT, OpT)
beh_ecg_resp_action_plv, plv_df_action = compute_ecg_resp_plv(beh_ecg_resp_df=beh_ecg_resp_action_filter, abbrev=abbrev)
beh_ecg_resp_tone_plv, plv_df_tone = compute_ecg_resp_plv(beh_ecg_resp_df=beh_ecg_resp_tone_filter, abbrev=abbrev_tone)

Mean PLV across participants and conditions for act trials: 0.1306
Mean PLV across participants and conditions for tone trials: 0.1180


In [21]:
# t-test of PLVs across BasA and OpA conditions
plv_df_action_wide = plv_df_action.pivot(index='subjID', columns='condition')['PLV'] 
plv_action_ttest = pg.ttest(plv_df_action_wide['BasA'], plv_df_action_wide['OpA'], paired=True, alternative='two-sided')

print(plv_action_ttest)

               T  dof alternative     p_val           CI95   cohen_d  \
T_test  0.125448   40   two-sided  0.900798  [-0.03, 0.03]  0.026937   

           power  BF10  
T_test  0.053254  0.17  


In [22]:
# t-test of PLVs across BasT and OpT conditions
plv_df_tone_wide = plv_df_tone.pivot(index='subjID', columns='condition')['PLV'] 
plv_tone_ttest = pg.ttest(plv_df_tone_wide['BasT'], plv_df_tone_wide['OpT'], paired=True, alternative='two-sided')

print(plv_tone_ttest)

               T  dof alternative     p_val           CI95   cohen_d  \
T_test -0.685118   40   two-sided  0.497221  [-0.03, 0.02]  0.139149   

           power  BF10  
T_test  0.140135  0.21  


In [ ]:
# Plot cardio-respiratory PLVs at action/tone onset across Baseline and Operant conditions

from matplotlib.colors import to_rgba

palette_action = [to_rgba(colpalette[1], 0.5), to_rgba(colpalette[0], 0.5)]
palette_tone = [to_rgba(colpalette[3], 0.5), to_rgba(colpalette[2], 0.5)]

# Plot cardio-respiratory PLVs at action onset between BasA and OpA
plt.figure(figsize=(12,6), dpi=300)
plt.subplot(1,2,1)
sns.boxplot(data=plv_df_action, x='condition', y='PLV', 
            hue='condition', palette=palette_action)
sns.stripplot(data=plv_df_action, x='condition', y='PLV', color='k', alpha=0.5, jitter=True)
#plt.title("Cardio-Respiratory Phase-Locking Value (PLV) at Action Onset")
plt.ylabel("PLV", fontsize='x-large')
plt.xlabel("Condition", fontsize='x-large')
plt.ylim(0, 0.4)

# Plot cardio-respiratory PLVs at tone onset between BasT and OpT
plt.subplot(1,2,2)
sns.boxplot(data=plv_df_tone, x='condition', y='PLV', 
            hue='condition', palette=palette_tone)
sns.stripplot(data=plv_df_tone, x='condition', y='PLV', color='k', alpha=0.5, jitter=True)
#plt.title("Cardio-Respiratory Phase-Locking Value (PLV) at Tone Onset")
plt.ylabel("PLV", fontsize='x-large')
plt.xlabel("Condition", fontsize='x-large')
plt.ylim(0, 0.4)
sns.despine()

plt.tight_layout()
plt.show()

## **4. R-R interval changes across respiratory cycle [Suppl. Results 1.2.4]**

This section provides descriptives for the cardio-respiratory interaction, namely changes in R-R interval duration across the respiratory cycle, which are analysed using a rmANOVA with 8 bins (45° each) relative to the first 0-45° bin. This is reported in Supplementary Results 1.2.4, with corresponding Figure S3c. 

In [24]:
##### Changes in R-R intervals across respiratory cycle #####

beh_ecg_resp_action_filter = beh_ecg_resp_action[beh_ecg_resp_action[column_map['condition']].isin(['BasA', 'OpA'])].copy()
beh_ecg_resp_tone_filter = beh_ecg_resp_tone[beh_ecg_resp_tone[column_map['condition']].isin(['BasT', 'OpT'])].copy()

# Bin respiratory phase at action onset into 8 segments (i.e., 45° bins)
beh_ecg_resp_action_filter['resp_bin'] = pd.cut(
    beh_ecg_resp_action_filter['event_resp_rad_act'],
    bins=np.linspace(0, 2*np.pi, 9),  # 8 bins
    labels=False, include_lowest=True)

# Compute mean R-R interval duration per subject per bin
rr_change_resp = beh_ecg_resp_action_filter.groupby(['subjID', 'resp_bin'])['RR_s_act'].mean().reset_index(name='RR_mean')

# Normalize R-R interval change to first bin 0-45°
baseline = rr_change_resp[rr_change_resp['resp_bin'] == 0].rename(columns={'RR_mean': 'RR_mean_baseline'})[['subjID', 'RR_mean_baseline']]
rr_change_resp = rr_change_resp.merge(baseline, on='subjID')
rr_change_resp['delta_RR_mean'] = (rr_change_resp['RR_mean'] - rr_change_resp['RR_mean_baseline']) * 1000       # convert to ms

In [ ]:
### FIGURE S3c ### 
# Plot changes in R-R intervals across respiratory cycle
bin_edges = np.linspace(0, 360, 9)  # 8 bins
bin_labels = [f"{int(bin_edges[i])}°\n-{int(bin_edges[i+1])}°" for i in range(len(bin_edges)-1)]

plt.figure(figsize=(6,6), dpi=300)
sns.boxplot(data=rr_change_resp, x='resp_bin', y='delta_RR_mean', color='lightgrey', linecolor='k')
plt.xticks(ticks=range(8), labels=bin_labels)
plt.xlabel('Respiration Phase (degrees)', fontsize='x-large')
plt.ylabel('Change in R-R intervals (ms) relative to 0-45° bin', fontsize='x-large')
#plt.title('R-R Interval Duration Across Respiratory Cycle', fontsize='x-large')
plt.ylim(-100, 200)
plt.tight_layout()
sns.despine()

figS3_3 = os.path.join(plotting_dir, "FigureS3c_RRchanges_respiration.svg")
plt.savefig(figS3_3, format="svg")
plt.show()

In [26]:
# rmANOVA of RR interval changes across respiratory cycle
rr_change_resp_anova = pg.rm_anova(
    dv=f'delta_RR_mean', within='resp_bin',
    subject=column_map['participant'], 
    data=rr_change_resp, detailed=True)
print(rr_change_resp_anova)

     Source             SS   DF           MS          F         p_unc  \
0  resp_bin   53009.460805    7  7572.780115  12.318103  1.082098e-13   
1     Error  167831.771055  273   614.768392        NaN           NaN   

      p_GG_corr       ng2       eps sphericity   W_spher       p_spher  
0  2.989871e-09  0.152152  0.616209      False  0.087677  1.920666e-08  
1           NaN       NaN       NaN        NaN       NaN           NaN  
